In [4]:
from google.colab import files
uploaded = files.upload()  # A button will appear — upload TSURU.xlsx

Saving TSURU.xlsx to TSURU.xlsx


In [5]:
import pandas as pd

# Load the Excel file
xl = pd.ExcelFile('TSURU.xlsx')

# See all sheet names
print("Total sheets:", len(xl.sheet_names))
print("\nAll sheet names:")
for i, name in enumerate(xl.sheet_names):
    print(f"  {i+1}. {name}")

Total sheets: 42

All sheet names:
  1. March 28
  2. March 31
  3. April 20
  4. April 27
  5. May 06
  6. May 13
  7. May 21
  8. May 28
  9. June 01
  10. June 25
  11. June 11
  12. June 17
  13. August 02
  14. July 05
  15. July 09
  16. July 17
  17. August 07
  18. August 22
  19. August 26
  20. September 07
  21. September 03
  22. September 11
  23. September 13
  24. September 19
  25. September 25
  26. October 03
  27. October 15
  28. October 26
  29. October 29
  30. November 02
  31. November 16
  32. November 05
  33. November 22
  34. November 27
  35. December 07
  36. December 13
  37. December 20
  38. December 25
  39. January 02 2022
  40. January 24 2022
  41. Test1 w batt
  42. Test2 wo batt


In [6]:
# Sheets to exclude (ground tests, not on-orbit)
EXCLUDE_SHEETS = ['Test1 w batt', 'Test2 wo batt']

# Only calibrated columns (no raw HEX)
CALIBRATED_COLS = [
    'Time stamp',
    'Tpy (°C)', 'Tpx (°C)', 'Tmz (°C)', 'Tmx (°C)', 'Tpz (°C)',
    'Vpy (mV)', 'Vpx (mV)', 'Vmz (mV)', 'Vmx (mV)', 'Vpz (mV)',
    'Ipy (mA)', 'Ipx (mA)', 'Imz (mA)', 'Imx (mA)', 'Ipz (mA)',
    'Vbat (V)', 'Ibatt(mA)', 'Tbatt (℃)'
]

dfs = []
for sheet in xl.sheet_names:
    if sheet in EXCLUDE_SHEETS:
        print(f"  SKIPPED: {sheet}")
        continue
    df = xl.parse(sheet)
    df = df[CALIBRATED_COLS]
    df['Date'] = sheet
    dfs.append(df)
    print(f"  Loaded: {sheet} → {len(df)} rows")

merged = pd.concat(dfs, ignore_index=True)
print(f"\n Final merged dataset: {merged.shape[0]} rows × {merged.shape[1]} columns")

  Loaded: March 28 → 1095 rows
  Loaded: March 31 → 359 rows
  Loaded: April 20 → 602 rows
  Loaded: April 27 → 605 rows
  Loaded: May 06 → 530 rows
  Loaded: May 13 → 595 rows
  Loaded: May 21 → 600 rows
  Loaded: May 28 → 587 rows
  Loaded: June 01 → 605 rows
  Loaded: June 25 → 596 rows
  Loaded: June 11 → 596 rows
  Loaded: June 17 → 592 rows
  Loaded: August 02 → 388 rows
  Loaded: July 05 → 593 rows
  Loaded: July 09 → 595 rows
  Loaded: July 17 → 596 rows
  Loaded: August 07 → 596 rows
  Loaded: August 22 → 589 rows
  Loaded: August 26 → 1146 rows
  Loaded: September 07 → 594 rows
  Loaded: September 03 → 590 rows
  Loaded: September 11 → 589 rows
  Loaded: September 13 → 594 rows
  Loaded: September 19 → 568 rows
  Loaded: September 25 → 594 rows
  Loaded: October 03 → 576 rows
  Loaded: October 15 → 601 rows
  Loaded: October 26 → 420 rows
  Loaded: October 29 → 179 rows
  Loaded: November 02 → 601 rows
  Loaded: November 16 → 1131 rows
  Loaded: November 05 → 591 rows
  Loade

In [7]:
print("=== DATASET OVERVIEW ===")
print(merged.head())

print("\n=== DATA TYPES ===")
print(merged.dtypes)

print("\n=== MISSING VALUES ===")
print(merged.isnull().sum())

print("\n=== BASIC STATISTICS ===")
print(merged.describe())

=== DATASET OVERVIEW ===
   Time stamp  Tpy (°C)  Tpx (°C)  Tmz (°C)  Tmx (°C)  Tpz (°C)  Vpy (mV)  \
0         0.0      4.63      6.19     -3.91     -0.81      8.29   1326.31   
1        11.3      4.30      5.96     -4.02     -1.25      7.85   1326.31   
2        22.6      3.63      5.41     -4.36     -1.58      7.30   1326.31   
3        33.9      3.19      5.19     -4.69     -2.03      6.85   1326.31   
4        45.2      2.63      4.97     -5.02     -2.36      6.52   1326.31   

   Vpx (mV)  Vmz (mV)  Vmx (mV)  Vpz (mV)  Ipy (mA)  Ipx (mA)  Imz (mA)  \
0   1272.89   1288.16   1318.68   1282.05     13.98       0.0      4.66   
1   1272.89   1288.16   1318.68   1282.05     13.98       0.0      4.66   
2   1272.89   1288.16   1320.21   1282.05     13.98       0.0      4.66   
3   1272.89   1286.63   1318.68   1282.05     13.98       0.0      9.32   
4   1271.37   1286.63   1318.68   1282.05     13.98       0.0      4.66   

   Imx (mA)  Ipz (mA)  Vbat (V)  Ibatt(mA)  Tbatt (℃)      Da

In [8]:
print("=== ISSUES FOUND ===")
print(f"1. Time stamp has 1,275 missing values → will drop these rows")
print(f"2. Vbat (V) min is 0.0 → impossible (battery can't be 0V) → remove these")
print(f"3. Voltages have 0.0 min → eclipse periods (valid, keep them)")
print(f"4. Ibatt max is 1200 mA → unusually high → check later")

# Fix 1: Drop rows where Time stamp is missing
merged = merged.dropna(subset=['Time stamp'])
print(f"\nAfter dropping missing timestamps: {len(merged)} rows")

# Fix 2: Drop rows where Vbat is 0 (sensor error)
merged = merged[merged['Vbat (V)'] > 0]
print(f"After removing Vbat=0 rows: {len(merged)} rows")

# Fix 3: Reset index
merged = merged.reset_index(drop=True)

print(f"\n Clean dataset: {merged.shape[0]} rows × {merged.shape[1]} columns")
print("\n=== MISSING VALUES AFTER CLEANING ===")
print(merged.isnull().sum())

=== ISSUES FOUND ===
1. Time stamp has 1,275 missing values → will drop these rows
2. Vbat (V) min is 0.0 → impossible (battery can't be 0V) → remove these
3. Voltages have 0.0 min → eclipse periods (valid, keep them)
4. Ibatt max is 1200 mA → unusually high → check later

After dropping missing timestamps: 23266 rows
After removing Vbat=0 rows: 23259 rows

 Clean dataset: 23259 rows × 20 columns

=== MISSING VALUES AFTER CLEANING ===
Time stamp    0
Tpy (°C)      0
Tpx (°C)      0
Tmz (°C)      0
Tmx (°C)      0
Tpz (°C)      0
Vpy (mV)      0
Vpx (mV)      0
Vmz (mV)      0
Vmx (mV)      0
Vpz (mV)      0
Ipy (mA)      0
Ipx (mA)      0
Imz (mA)      0
Imx (mA)      0
Ipz (mA)      0
Vbat (V)      0
Ibatt(mA)     0
Tbatt (℃)     0
Date          0
dtype: int64


In [9]:
import numpy as np

# === FEATURE ENGINEERING ===

# 1. Power per panel (P = V × I), converted from mW to µW
merged['Ppy (µW)'] = merged['Vpy (mV)'] * merged['Ipy (mA)']
merged['Ppx (µW)'] = merged['Vpx (mV)'] * merged['Ipx (mA)']
merged['Pmz (µW)'] = merged['Vmz (mV)'] * merged['Imz (mA)']
merged['Pmx (µW)'] = merged['Vmx (mV)'] * merged['Imx (mA)']
merged['Ppz (µW)'] = merged['Vpz (mV)'] * merged['Ipz (mA)']

# 2. Total solar power
merged['P_solar_total (µW)'] = (merged['Ppy (µW)'] + merged['Ppx (µW)'] +
                                 merged['Pmz (µW)'] + merged['Pmx (µW)'] +
                                 merged['Ppz (µW)'])

# 3. Battery power
merged['P_batt (µW)'] = merged['Vbat (V)'] * 1000 * merged['Ibatt(mA)']

# 4. Average panel temperature
merged['T_panel_avg (°C)'] = merged[['Tpy (°C)','Tpx (°C)',
                                      'Tmz (°C)','Tmx (°C)',
                                      'Tpz (°C)']].mean(axis=1)

# 5. Temperature difference battery vs panel
merged['ΔT_batt_panel (°C)'] = merged['Tbatt (℃)'] - merged['T_panel_avg (°C)']

# 6. Average panel voltage
merged['V_panel_avg (mV)'] = merged[['Vpy (mV)','Vpx (mV)',
                                      'Vmz (mV)','Vmx (mV)',
                                      'Vpz (mV)']].mean(axis=1)

# 7. Orbital phase encoding (92.6 min = 5556 seconds)
merged['sin_orbit'] = np.sin(2 * np.pi * merged['Time stamp'] / 5556)
merged['cos_orbit'] = np.cos(2 * np.pi * merged['Time stamp'] / 5556)

# === CLASSIFICATION LABELS ===

# Label 1: Eclipse detection (0 = sunlight, 1 = eclipse)
merged['Eclipse'] = (merged['P_solar_total (µW)'] < 10000).astype(int)

# Label 2: Charge state (1 = charging, 0 = discharging)
# From paper: Negative Ibatt = charging, Positive = discharging
merged['Charging'] = (merged['Ibatt(mA)'] < 0).astype(int)

# Label 3: Power level (Low / Medium / High)
merged['Power_Level'] = pd.qcut(merged['P_solar_total (µW)'],
                                 q=3,
                                 labels=['Low', 'Medium', 'High'])

print("=== FEATURE ENGINEERING COMPLETE ===")
print(f"Total columns now: {merged.shape[1]}")
print(f"\nNew columns added:")
new_cols = ['Ppy (µW)','Ppx (µW)','Pmz (µW)','Pmx (µW)','Ppz (µW)',
            'P_solar_total (µW)','P_batt (µW)','T_panel_avg (°C)',
            'ΔT_batt_panel (°C)','V_panel_avg (mV)','sin_orbit','cos_orbit',
            'Eclipse','Charging','Power_Level']
for col in new_cols:
    print(f"   {col}")

print(f"\n=== LABEL DISTRIBUTION ===")
print(f"\nEclipse (0=Sunlight, 1=Eclipse):")
print(merged['Eclipse'].value_counts())
print(f"\nCharging (1=Charging, 0=Discharging):")
print(merged['Charging'].value_counts())
print(f"\nPower Level:")
print(merged['Power_Level'].value_counts())

=== FEATURE ENGINEERING COMPLETE ===
Total columns now: 35

New columns added:
   Ppy (µW)
   Ppx (µW)
   Pmz (µW)
   Pmx (µW)
   Ppz (µW)
   P_solar_total (µW)
   P_batt (µW)
   T_panel_avg (°C)
   ΔT_batt_panel (°C)
   V_panel_avg (mV)
   sin_orbit
   cos_orbit
   Eclipse
   Charging
   Power_Level

=== LABEL DISTRIBUTION ===

Eclipse (0=Sunlight, 1=Eclipse):
Eclipse
0    23259
Name: count, dtype: int64

Charging (1=Charging, 0=Discharging):
Charging
1    13523
0     9736
Name: count, dtype: int64

Power Level:
Power_Level
Low       7753
Medium    7753
High      7753
Name: count, dtype: int64


In [10]:
# First let's understand the solar power distribution
print("=== SOLAR POWER ANALYSIS ===")
print(merged['P_solar_total (µW)'].describe())

print(f"\nHow many rows have P_solar_total = 0:")
print(len(merged[merged['P_solar_total (µW)'] == 0]))

print(f"\nHow many rows have P_solar_total < 50000:")
print(len(merged[merged['P_solar_total (µW)'] < 50000]))

print(f"\nHow many rows have P_solar_total < 100000:")
print(len(merged[merged['P_solar_total (µW)'] < 100000]))

print(f"\nHow many rows have P_solar_total < 500000:")
print(len(merged[merged['P_solar_total (µW)'] < 500000]))

print(f"\nValue counts of P_solar_total in ranges:")
bins = [0, 50000, 200000, 500000, 1000000, 5000000]
labels = ['0-50k', '50k-200k', '200k-500k', '500k-1M', '1M+']
merged['temp_range'] = pd.cut(merged['P_solar_total (µW)'], bins=bins, labels=labels)
print(merged['temp_range'].value_counts().sort_index())
merged = merged.drop('temp_range', axis=1)

=== SOLAR POWER ANALYSIS ===
count    2.325900e+04
mean     1.079542e+06
std      9.073104e+05
min      3.019181e+04
25%      3.618765e+04
50%      1.210447e+06
75%      1.585577e+06
max      4.378088e+06
Name: P_solar_total (µW), dtype: float64

How many rows have P_solar_total = 0:
0

How many rows have P_solar_total < 50000:
7722

How many rows have P_solar_total < 100000:
7796

How many rows have P_solar_total < 500000:
7995

Value counts of P_solar_total in ranges:
temp_range
0-50k         7722
50k-200k       140
200k-500k      133
500k-1M        935
1M+          14329
Name: count, dtype: int64


In [11]:
# Fix Eclipse label using correct threshold
# Clear gap visible: below 50,000 µW = eclipse, above = sunlight
merged['Eclipse'] = (merged['P_solar_total (µW)'] < 50000).astype(int)

print("=== FIXED LABEL DISTRIBUTION ===")
print(f"\nEclipse (0=Sunlight, 1=Eclipse):")
print(merged['Eclipse'].value_counts())
print(f"\nPercentage:")
print(merged['Eclipse'].value_counts(normalize=True).mul(100).round(1))

print(f"\nCharging (1=Charging, 0=Discharging):")
print(merged['Charging'].value_counts())
print(f"\nPercentage:")
print(merged['Charging'].value_counts(normalize=True).mul(100).round(1))

print(f"\nPower Level:")
print(merged['Power_Level'].value_counts())

print("\n=== PHYSICAL VERIFICATION ===")
# When in eclipse, satellite should mostly be charging (Ibatt < 0)
eclipse_rows = merged[merged['Eclipse'] == 1]
sunlight_rows = merged[merged['Eclipse'] == 0]
print(f"\nDuring Eclipse → Charging %: {(eclipse_rows['Charging'].sum()/len(eclipse_rows)*100):.1f}%")
print(f"During Sunlight → Charging %: {(sunlight_rows['Charging'].sum()/len(sunlight_rows)*100):.1f}%")
print("\n(During eclipse battery should discharge → low charging %)")
print("(During sunlight battery should charge → high charging %)")

=== FIXED LABEL DISTRIBUTION ===

Eclipse (0=Sunlight, 1=Eclipse):
Eclipse
0    15537
1     7722
Name: count, dtype: int64

Percentage:
Eclipse
0    66.8
1    33.2
Name: proportion, dtype: float64

Charging (1=Charging, 0=Discharging):
Charging
1    13523
0     9736
Name: count, dtype: int64

Percentage:
Charging
1    58.1
0    41.9
Name: proportion, dtype: float64

Power Level:
Power_Level
Low       7753
Medium    7753
High      7753
Name: count, dtype: int64

=== PHYSICAL VERIFICATION ===

During Eclipse → Charging %: 0.0%
During Sunlight → Charging %: 87.0%

(During eclipse battery should discharge → low charging %)
(During sunlight battery should charge → high charging %)


In [12]:
# Save the final cleaned dataset as CSV
merged.to_csv('TSURU_cleaned.csv', index=False)
print(" Saved: TSURU_cleaned.csv")

# Also save test sheets separately for bonus analysis
test1 = xl.parse('Test1 w batt')
test2 = xl.parse('Test2 wo batt')
test1.to_csv('TSURU_test1_wbatt.csv', index=False)
test2.to_csv('TSURU_test2_wobatt.csv', index=False)
print(" Saved: TSURU_test1_wbatt.csv")
print(" Saved: TSURU_test2_wobatt.csv")

# Final summary
print("\n=== FINAL DATASET SUMMARY ===")
print(f"Total rows          : {len(merged)}")
print(f"Total columns       : {merged.shape[1]}")
print(f"Date range          : {merged['Date'].iloc[0]} → {merged['Date'].iloc[-1]}")
print(f"Total sheets merged : 40 (on-orbit only)")
print(f"Excluded sheets     : Test1 w batt, Test2 wo batt")
print(f"\nFeature columns     : 18 original + 12 engineered = 30")
print(f"Classification labels: Eclipse, Charging, Power_Level")
print(f"\n Data preparation COMPLETE")

# Download the file to your computer
from google.colab import files
files.download('TSURU_cleaned.csv')

 Saved: TSURU_cleaned.csv
 Saved: TSURU_test1_wbatt.csv
 Saved: TSURU_test2_wobatt.csv

=== FINAL DATASET SUMMARY ===
Total rows          : 23259
Total columns       : 35
Date range          : March 28 → January 24 2022
Total sheets merged : 40 (on-orbit only)
Excluded sheets     : Test1 w batt, Test2 wo batt

Feature columns     : 18 original + 12 engineered = 30
Classification labels: Eclipse, Charging, Power_Level

 Data preparation COMPLETE


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>